In [1]:
!pip install -q transformers datasets accelerate evaluate scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.6 MB/s eta 0:00:00


In [2]:
import re
import numpy as np
import pandas as pd
import torch

from google.colab import files

from datasets import Dataset
from datasets import DatasetDict

from sklearn.model_selection import GroupShuffleSplit

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support
)

from transformers import (

    AutoTokenizer,
    AutoModelForSequenceClassification,

    TrainingArguments,
    Trainer,

    EarlyStoppingCallback,
    DataCollatorWithPadding
)

In [3]:
SEED = 42

np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [4]:
uploaded = files.upload()

df = pd.read_csv("full_dataset.csv")

print("Dataset Shape:", df.shape)

Saving full_dataset.csv to full_dataset.csv
Dataset Shape: (1690, 6)


In [5]:
df = df[["text", "label", "group"]]

In [6]:
df = df.dropna(subset=["text"])


In [7]:
df = df.drop_duplicates(subset=["text"])

In [8]:
def clean_text(text):

    text = str(text).lower()

    text = re.sub(r"\s+", " ", text)

    text = text.strip()

    return text

df["text"] = df["text"].apply(clean_text)

print(df.head())

                                                text  label  \
0  1.0 do not buy or use this gamepad! no stars! ...      0   
1  49 19 r3tp2ynw3x41d8 daughter makes fun of bea...      1   
2  14 4 r202zzkbabcxd7 this is another of visiona...      1   
3  5.0 read this as a teenager, reread it as an a...      0   
4  33 7 r33ypav3d4qu2p if you pay 250 for this bl...      1   

                     group  
0  5_18_R3VMBP3ZGTBC6G.txt  
1              sarcasm_382  
2               sarcasm_92  
3  36_9_R1V3CD3WQFT5S7.txt  
4              sarcasm_248  


In [9]:
groups = df["group"]

gss = GroupShuffleSplit(

    n_splits=1,
    test_size=0.20,
    random_state=SEED
)

train_idx, temp_idx = next(
    gss.split(df, groups=groups)
)

train_df = df.iloc[train_idx]
temp_df = df.iloc[temp_idx]

gss2 = GroupShuffleSplit(

    n_splits=1,
    test_size=0.50,
    random_state=SEED
)

val_idx, test_idx = next(

    gss2.split(
        temp_df,
        groups=temp_df["group"]
    )
)

val_df = temp_df.iloc[val_idx]
test_df = temp_df.iloc[test_idx]

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

Train: 1352
Validation: 169
Test: 169


In [10]:
dataset = DatasetDict({

    "train": Dataset.from_pandas(
        train_df[["text", "label"]]
    ),

    "validation": Dataset.from_pandas(
        val_df[["text", "label"]]
    ),

    "test": Dataset.from_pandas(
        test_df[["text", "label"]]
    )
})


In [11]:
model_name = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(
    model_name
)

model = AutoModelForSequenceClassification.from_pretrained(

    model_name,

    num_labels=2
)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [21]:
def tokenize_function(example):

    return tokenizer(

        example["text"],

        truncation=True,

        padding=False,

        max_length=192
    )

tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True
)

Map:   0%|          | 0/1352 [00:00<?, ? examples/s]

Map:   0%|          | 0/169 [00:00<?, ? examples/s]

Map:   0%|          | 0/169 [00:00<?, ? examples/s]

In [22]:
data_collator = DataCollatorWithPadding(tokenizer)

In [14]:
def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(
        logits,
        axis=-1
    )

    precision, recall, f1, _ = precision_recall_fscore_support(

        labels,

        predictions,

        average="macro"
    )

    accuracy = accuracy_score(
        labels,
        predictions
    )

    return {

        "accuracy": accuracy,

        "macro_precision": precision,

        "macro_recall": recall,

        "macro_f1": f1
    }

In [36]:
training_args = TrainingArguments(

    output_dir="./results",

    eval_strategy="epoch",

    save_strategy="epoch",

    load_best_model_at_end=True,

    metric_for_best_model="macro_f1",

    greater_is_better=True,

    learning_rate=1.5e-5,

    weight_decay=0.01,

    warmup_ratio=0.1,

    lr_scheduler_type="cosine",

    num_train_epochs=2,

    per_device_train_batch_size=8,

    per_device_eval_batch_size=8,

    gradient_accumulation_steps=2,

    fp16=torch.cuda.is_available(),

    max_grad_norm=1.0,

    logging_steps=5,
    logging_strategy="steps",

    save_total_limit=2,

    seed=SEED,

    report_to="none"
)


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [37]:
trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=tokenized_dataset["train"],

    eval_dataset=tokenized_dataset["validation"],

    data_collator=data_collator,

    compute_metrics=compute_metrics,

    callbacks=[

        EarlyStoppingCallback(
            early_stopping_patience=1
        )
    ]
)

In [38]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Macro Precision,Macro Recall,Macro F1
1,0.403251,0.420631,0.899408,0.898053,0.896689,0.897338
2,0.200331,0.499033,0.881657,0.897304,0.867937,0.875699


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=170, training_loss=0.2668399879161049, metrics={'train_runtime': 77.2076, 'train_samples_per_second': 35.022, 'train_steps_per_second': 2.202, 'total_flos': 134201893657152.0, 'train_loss': 0.2668399879161049, 'epoch': 2.0})

In [19]:
torch.save(
    model.state_dict(),
    "distilbert-base-uncased_weights.pth"
)

label
1    873
0    817
Name: count, dtype: int64


In [ ]:
from google.colab import files

files.download("distilbert-base-uncased_weights.pth")